In [1]:
import pandas as pd
df = pd.read_csv('data/bank-full.csv', sep=';')
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [2]:
import numpy as np

df['subscribed'] = (df['y'] == 'yes').astype(int)
BASELINE = df['subscribed'].mean()

def conversion_table(data, by):
    """按 by 列分组，计算转化率、95% Wilson 置信区间和 lift，按区间下限排序"""
    t = data.groupby(by, observed=True)['subscribed'].agg(n='size', conversions='sum').reset_index()
    n, p, z = t['n'], t['conversions'] / t['n'], 1.96
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    t['rate'] = p
    t['ci_low'] = center - margin
    t['ci_high'] = center + margin
    t['lift'] = p / BASELINE
    return t.sort_values('ci_low', ascending=False).round(3)

conversion_table(df, 'job')

,job,n,conversions,rate,ci_low,ci_high,lift
8,student,938,269,0.287,0.259,0.317,2.451
5,retired,2264,516,0.228,0.211,0.246,1.948
10,unemployed,1303,202,0.155,0.136,0.176,1.325
4,management,9458,1301,0.138,0.131,0.145,1.176
0,admin.,5171,631,0.122,0.113,0.131,1.043
9,technician,7597,840,0.111,0.104,0.118,0.945
6,self-employed,1579,187,0.118,0.103,0.135,1.012
11,unknown,288,34,0.118,0.086,0.160,1.009
7,services,4154,369,0.089,0.081,0.098,0.759
3,housemaid,1240,109,0.088,0.073,0.105,0.751
